# CHUYÊN ĐỀ TOÀN DIỆN: CÁC KỸ THUẬT TỐI ƯU HÓA SPARK & DELTA LAKE DÀNH CHO DATA ENGINEER (PROD-READY)

Notebook này được thiết kế theo tiêu chuẩn thực nghiệm khắt khe trong môi trường Enterprise:
- **Loại bỏ sai số Cold-Start (Làm nóng JVM / Spark Warm-up trước khi đo):** Đảm bảo kết quả benchmark phản ánh đúng 100% hiệu năng thuật toán.
- **So sánh Đối Đầu (Before vs After):** Mỗi kỹ thuật đều có 2 kịch bản chạy song song để bạn thấy rõ sự khác biệt về **Physical Plan**, **Thời gian thực thi**, và **Phân bố dữ liệu trên Partition**.
- **Đầy đủ 6 Kỹ Thuật Trọng Điểm:**
  1. **Broadcast Join vs Sort-Merge Join:** Triệt tiêu Network Shuffle.
  2. **Adaptive Query Execution (AQE):** Tự động gom Partition lúc runtime.
  3. **Xử lý Data Skew bằng kỹ thuật Salting:** Minh họa trực quan phân bố dòng trên từng partition.
  4. **`coalesce()` vs `repartition()`:** Tối ưu hóa ghi file và loại bỏ Shuffle.
  5. **Delta Lake Compaction & Z-ORDER:** Giải quyết Small Files và Data Skipping.
  6. **Delta Lake `VACUUM`:** Dọn dẹp file mồ côi kiểm soát chi phí storage.

## BƯỚC 0: KHỞI ĐỘNG VÀ LÀM NÓNG MÁY ẢO JVM (SPARK WARM-UP)

> **Tại sao cần Warm-up?** 
> Khi Spark vừa khởi động, các Thread Pool của Executor chưa được cấp phát, JIT Compiler của Java chưa nạp Bytecode. Lệnh đầu tiên luôn bị trễ thêm 1-2 giây vì "Cold Start". Ta chạy một lệnh giả lập để JVM sẵn sàng trước khi thực hiện các bài benchmark.

In [ ]:
import time
from pyspark.sql import functions as F
from pyspark.sql.types import *

print("--- ĐANG THỰC HIỆN SPARK & JVM WARM-UP ---")
t_start = time.time()
_ = spark.range(0, 50000).groupBy().count().collect()
print(f"-> JVM và Spark Context đã sẵn sàng (Thời gian khởi động: {time.time() - t_start:.2f}s)!")

# Hàm tiện ích đo thời gian chạy công bằng (chạy 1 lần làm ấm, lấy trung bình 2 lần sau)
def benchmark_action(action_fn, name="Action", runs=2):
    # Lần 1: Chạy để đưa dữ liệu vào pipeline cache nếu có
    action_fn()
    # Các lần sau: Đo đạc chính thức
    times = []
    for _ in range(runs):
        t0 = time.time()
        action_fn()
        times.append(time.time() - t0)
    avg_time = sum(times) / len(times)
    print(f"[{name}] Thời gian trung bình: {avg_time:.4f} giây")
    return avg_time


## KỸ THUẬT 1: BROADCAST HASH JOIN VS SORT-MERGE JOIN

### Thử nghiệm chuẩn xác:
Mặc định Spark tự động Broadcast các bảng < 10MB. Để thấy rõ sự khác biệt của **Sort-Merge Join (Shuffle cả 2 bảng)** và **Broadcast Join (Không Shuffle)**, ta sẽ:
1. Tắt tự động Broadcast (`autoBroadcastJoinThreshold = -1`) $ightarrow$ Ép Spark dùng **Sort-Merge Join**.
2. Bật lại Hint `broadcast()` $ightarrow$ Ép Spark dùng **Broadcast Hash Join**.

In [ ]:
# Tạo 1 bảng Fact 200,000 dòng và 1 bảng Dim 1,000 dòng
df_fact = spark.range(0, 200000) \
    .withColumn("customer_id", F.concat(F.lit("CUST_"), (F.col("id") % 1000).cast("string"))) \
    .withColumn("revenue", F.rand() * 1000)

df_dim = spark.range(0, 1000) \
    .withColumn("customer_id", F.concat(F.lit("CUST_"), F.col("id").cast("string"))) \
    .withColumn("customer_group", F.when(F.col("id") % 2 == 0, "RETAIL").otherwise("WHOLESALE"))

# ==============================================================================
# 1.1 ÉP BUỘC SORT-MERGE JOIN (BẰNG CÁCH TẮT AUTO BROADCAST)
# ==============================================================================
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1) # Tắt broadcast tự động
df_smj = df_fact.join(df_dim, "customer_id")

print("=== 1.1 KẾ HOẠCH SORT-MERGE JOIN (CÓ SHUFFLE CẢ 2 BẢNG) ===")
df_smj.explain(False)
t_smj = benchmark_action(lambda: df_smj.count(), name="Sort-Merge Join")

# ==============================================================================
# 1.2 DÙNG HINT BROADCAST() (TRIỆT TIÊU TOÀN BỘ SHUFFLE)
# ==============================================================================
df_bhj = df_fact.join(F.broadcast(df_dim), "customer_id")

print("\n=== 1.2 KẾ HOẠCH BROADCAST HASH JOIN (KHÔNG SHUFFLE) ===")
df_bhj.explain(False)
t_bhj = benchmark_action(lambda: df_bhj.count(), name="Broadcast Hash Join")

# Khôi phục lại ngưỡng mặc định 10MB của Spark
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", 10485760)

print(f"\n-> KẾT QUẢ: Broadcast Join nhanh hơn gấp {t_smj / t_bhj:.2f} lần so với Sort-Merge Join!")


## KỸ THUẬT 2: ADAPTIVE QUERY EXECUTION (AQE) - TỰ ĐỘNG GỘP SHUFFLE PARTITIONS

### Vấn đề Partition Mặc Định:
Mặc định Spark đặt `spark.sql.shuffle.partitions = 200`. Với các truy vấn tổng hợp vừa và nhỏ, 200 partition khiến Spark chia nhỏ dữ liệu thành 200 task tí hon, làm nghẽn Scheduler.

### Cơ chế AQE:
Khi bật AQE, Spark tự quan sát kích thước thực tế sau khi Shuffle và tự động gộp (Coalesce) 200 partition xuống còn đúng số lượng cần thiết (ví dụ 4-8 partition)!

In [ ]:
# 2.1 TRƯỜNG HỢP KHÔNG CÓ AQE (TẮT AQE)
spark.conf.set("spark.sql.adaptive.enabled", "false")
df_no_aqe = df_fact.groupBy("customer_id").agg(F.sum("revenue").alias("total"))
df_no_aqe.collect() # Trigger action
partitions_no_aqe = df_no_aqe.rdd.getNumPartitions()
print(f"[KHÔNG BẬT AQE] Số lượng Shuffle Partitions bị cố định: {partitions_no_aqe} partitions (Gây lãng phí tài nguyên)")

# 2.2 TRƯỜNG HỢP BẬT AQE TỐI ƯU
spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled", "true")
spark.conf.set("spark.sql.adaptive.coalescePartitions.minPartitionSize", "1MB")

df_with_aqe = df_fact.groupBy("customer_id").agg(F.sum("revenue").alias("total"))
df_with_aqe.collect()
partitions_aqe = df_with_aqe.rdd.getNumPartitions()
print(f"[ĐÃ BẬT AQE]   Số lượng Shuffle Partitions tự động gộp tối ưu: {partitions_aqe} partitions!")

print("\n--- Kế hoạch AdaptiveQuery (Notice: CoalescedShuffleSpec): ---")
df_with_aqe.explain(False)


## KỸ THUẬT 3: XỬ LÝ LỆCH DỮ LIỆU (DATA SKEW) BẰNG KỸ THUẬT SALTING

### Minh họa hiện tượng nghẽn cổ chai khi JOIN:
- **Giả lập:** 90% đơn hàng thuộc về 1 khách hàng VIP duy nhất (`CUST_SKEWED`).
- **Khi Join thường (Chưa Salting):** Toàn bộ 90,000 dòng có key `CUST_SKEWED` bị ép về chung **1 Executor/Partition** duy nhất $\rightarrow$ Gây nghẽn cổ chai (Straggler) hoặc sập bộ nhớ (OOM).
- **Kỹ thuật Salting (Có Salting):**
  1. **Bảng Lớn:** Ghép số ngẫu nhiên `0..3` vào key `CUST_SKEWED` $\rightarrow$ Xé nhỏ dữ liệu thành 4 nhóm key (`CUST_SKEWED_0`, `CUST_SKEWED_1`, `CUST_SKEWED_2`, `CUST_SKEWED_3`) để chia đều sang 4 Partition.
  2. **Bảng Nhỏ:** Nhân bản (`explode`) mỗi bản ghi thành đủ 4 salt (`0..3`) để ghép khớp với mọi salt ngẫu nhiên từ bảng lớn.
- Dùng hàm `spark_partition_id()` để soi trực tiếp sự phân bổ dữ liệu **KẾT QUẢ PHÉP JOIN** trước và sau khi Salting!

In [ ]:
# 1. Tắt Broadcast Join & Tắt AQE Coalesce để thấy rõ sự phân bổ Shuffle Partition thực sự
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1)
spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled", "false")

# 2. Tạo tập dữ liệu bị Skew nặng (90% dòng là CUST_SKEWED)
df_skewed_data = spark.range(0, 100000) \
    .withColumn("customer_id", F.when(F.col("id") < 90000, "CUST_SKEWED").otherwise(F.concat(F.lit("CUST_"), F.col("id")))) \
    .withColumn("amount", F.rand() * 500)

df_lookup_dim = spark.createDataFrame([
    ("CUST_SKEWED", "VIP Super Client"),
    ("CUST_95000", "Regular Client")
], ["customer_id", "tier"])

# ==============================================================================
# 3.1 THỬ NGHIỆM JOIN KHI CHƯA SALTING (1 PARTITION BỊ QUÁ TẢI NẶNG)
# ==============================================================================
df_unbalanced_data = df_skewed_data.repartition(4, "customer_id")
df_joined_unbalanced = df_unbalanced_data.join(df_lookup_dim, on="customer_id", how="inner")
print("=== 3.1 PHÂN BỐ DÒNG KHI JOIN CHƯA SALTING (1 PARTITION ÔM 90,000+ DÒNG) ===")
df_joined_unbalanced.groupBy(F.spark_partition_id().alias("partition_id")).count().orderBy("partition_id").show()

# ==============================================================================
# 3.2 THỬ NGHIỆM JOIN SAU KHI ÁP DỤNG KỸ THUẬT SALTING
# ==============================================================================
SALT_FACTOR = 4

# Bảng lớn: ghép salt ngẫu nhiên 0..3 và REPARTITION theo salted_key
df_skewed_salted = df_skewed_data \
    .withColumn("salt", (F.rand() * SALT_FACTOR).cast("int")) \
    .withColumn("salted_key", F.concat_ws("_", F.col("customer_id"), F.col("salt"))) \
    .repartition(4, "salted_key")

# Bảng nhỏ: nhân bản explode đủ 4 salt
df_dim_salted = df_lookup_dim \
    .withColumn("salt_array", F.array([F.lit(i) for i in range(SALT_FACTOR)])) \
    .withColumn("salt", F.explode("salt_array")) \
    .withColumn("salted_key", F.concat_ws("_", F.col("customer_id"), F.col("salt"))) \
    .drop("salt_array")

# PHÉP JOIN THỰC TẾ THEO SALTED_KEY
df_joined_salted = df_skewed_salted.join(df_dim_salted, on="salted_key", how="inner")

print("=== 3.2 PHÂN BỐ DÒNG KHI JOIN CÓ SALTING (DỮ LIỆU ĐƯỢC CHIA ĐỀU CHO 4 PARTITION) ===")
df_joined_salted.groupBy(F.spark_partition_id().alias("partition_id")).count().orderBy("partition_id").show()

# Khôi phục lại cấu hình mặc định của Spark
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", 10485760)
spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled", "true")


## KỸ THUẬT 4: `coalesce()` VS `repartition()` KHI GHI FILE

### 1. Tại sao cần hiểu rõ sự khác biệt?
- **`coalesce(n)`:** Chỉ gom các partition kế cận trên cùng node $\rightarrow$ **0% Shuffle (Tuyệt đối không có toán tử Exchange)**. Cực kỳ nhanh khi muốn giảm số file trước khi ghi ra đĩa.
- **`repartition(n)`:** Xáo trộn toàn bộ dữ liệu qua mạng (**Full Shuffle - Toán tử Exchange**) $\rightarrow$ Tốn CPU, I/O đĩa và băng thông mạng. Chỉ dùng khi muốn tăng độ song song hoặc phân bổ lại dữ liệu theo Hash Key.

### 2. Lưu ý tinh tế về bộ tối ưu Catalyst:
Nếu tạo dữ liệu bằng `spark.range(...).repartition(16)`, Catalyst sẽ tự động gộp hai lệnh repartition liên tiếp (`repartition(16)` + `repartition(4)` $\rightarrow$ `repartition(4)`). 
Vì vậy, để kiểm chứng chuẩn xác, ta tạo 16 partition **ngay từ nguồn** bằng `spark.range(0, 3000000, 1, 16)` để thấy rõ: `coalesce` hoàn toàn không có toán tử `Exchange`, trong khi `repartition` bắt buộc phải tạo `Exchange` qua mạng!

In [ ]:
# 1. Khởi tạo DataFrame có sẵn 16 partitions tự nhiên từ nguồn (không dùng lệnh .repartition(16))
# Cú pháp: spark.range(start, end, step, numPartitions)
df_wide = spark.range(0, 3000000, 1, 16)
print(f"Số partition ban đầu từ nguồn: {df_wide.rdd.getNumPartitions()}")

# 4.1 SOI PHYSICAL PLAN ĐỂ THẤY SỰ KHÁC BIỆT VỀ SHUFFLE (EXCHANGE)
print("\n--- Physical Plan của coalesce(4) [HOÀN TOÀN KHÔNG CÓ EXCHANGE (0% SHUFFLE)]: ---")
df_wide.coalesce(4).explain(False)

print("\n--- Physical Plan của repartition(4) [CÓ TOÁN TỬ EXCHANGE ROUNDROBIN (100% SHUFFLE)]: ---")
df_wide.repartition(4).explain(False)

# 4.2 ĐO THỜI GIAN THỰC THI (KÈM PHÉP TÍNH TOÁN DỮ LIỆU THỰC TẾ)
# Thêm phép tính để CPU và Memory phải xử lý dữ liệu
df_proc = df_wide.withColumn("val", F.col("id") * 2)

print("\n--- Đang đo thời gian thực thi... ---")
t_coalesce = benchmark_action(lambda: df_proc.coalesce(4).filter("val > 0").count(), name="coalesce(4)")
t_repartition = benchmark_action(lambda: df_proc.repartition(4).filter("val > 0").count(), name="repartition(4)")

print(f"\n-> KẾT LUẬN & ĐÚC KẾT THỰC TẾ:")
print(f"1. Nhìn vào Physical Plan: `coalesce(4)` loại bỏ hoàn toàn toán tử Exchange. Toàn bộ 16 partition được gộp tại chỗ (Local Merge) mà không gửi qua mạng.")
print(f"2. `repartition(4)` bắt buộc phải đẩy toàn bộ 3 triệu dòng qua mạng (Network Shuffle) để chia lại theo thuật toán RoundRobin.")
print(f"3. Khi ghi file (.write.parquet / .saveAsTable), LUÔN LUÔN ưu tiên `coalesce(n)` để giảm số file nhỏ, tránh làm nghẽn Cluster bởi các tác vụ Shuffle không cần thiết.")


## KỸ THUẬT 5: DELTA LAKE COMPACTION & Z-ORDER CLUSTERING

### Thử nghiệm giải quyết Small Files Problem:
1. Giả lập ghi nhiều mẻ nhỏ $ightarrow$ Sinh ra nhiều file Parquet tí hon.
2. Chạy `OPTIMIZE ... ZORDER BY (customer_id)` $ightarrow$ Gom file và sắp xếp theo đường cong lấp đầy không gian Z-Order để tối ưu Data Skipping.

In [ ]:
tbl_opt_demo = "opt_compaction_zorder_demo"

# Giả lập ghi 8 partition nhỏ ra Delta Lake
spark.range(0, 50000) \
    .withColumn("customer_id", F.concat(F.lit("CUST_"), (F.col("id") % 500).cast("string"))) \
    .withColumn("order_val", F.rand() * 1000) \
    .repartition(8) \
    .write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(tbl_opt_demo)

print("=== TRƯỚC KHI TỐI ƯU HÓA ===")
spark.sql(f"DESCRIBE DETAIL {tbl_opt_demo}").select("numFiles", "sizeInBytes").show()

# Thực hiện Gom file (Compaction) và Sắp xếp Z-Order
print("--- Đang thực thi OPTIMIZE kèm Z-ORDER BY (customer_id)... ---")
spark.sql(f"OPTIMIZE {tbl_opt_demo} ZORDER BY (customer_id)").show(truncate=False)

print("=== SAU KHI TỐI ƯU HÓA (CÁC FILE NHỎ ĐÃ GỘP THÀNH FILE CHUẨN) ===")
spark.sql(f"DESCRIBE DETAIL {tbl_opt_demo}").select("numFiles", "sizeInBytes").show()


## KỸ THUẬT 6: QUẢN LÝ DỌN DẸP BẢN GHI RÁC BẰNG DELTA LAKE `VACUUM`

Khi dữ liệu bị sửa (`UPDATE`) hoặc gộp (`OPTIMIZE`), các file Parquet cũ trở thành "file rác" (tombstone). Sau 7-30 ngày, lệnh **`VACUUM`** sẽ xóa vĩnh viễn các file này để giải phóng dung lượng đĩa.

In [ ]:
# Tắt ngưỡng an toàn 7 ngày (Chỉ dùng cho mục đích Demo/Lab để thấy file bị xóa ngay lập tức):
spark.conf.set("spark.databricks.delta.vacuum.parallelDelete.enabled", "true")
spark.conf.set("spark.databricks.delta.retentionDurationCheck.enabled", "false")

print(f"--- ĐANG THỰC HIỆN VACUUM TRÊN BẢNG {tbl_opt_demo} ---")
spark.sql(f"VACUUM {tbl_opt_demo} RETAIN 0 HOURS").show(truncate=False)

print("Đã dọn sạch toàn bộ các file Parquet mồ côi cũ! Bảng đạt trạng thái lưu trữ gọn gàng nhất.")
